# Evaluacion Modulo 1

## Ejercicio 1: Búsqueda A* en un campus universitario


### Integrantes:

- Hellen Yanes Doria 
- Mariana Sanchez 
- Andres Echeverri 


# MAPA

In [5]:
Campus = [
# x:  0    1    2     3    4     5     6    7    8     9    10   11    12   13   14    15   16
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "],  # y=0
    [" ", "#", " ",  " ", "#",  " ",  " ", "#", " ",  " ", "#", " ",  " ", "#", " ",  " ", " "],  # y=1

    [" ", " ", "B30"," ", " ", "B31"," ", " ", "B32"," ", " ", "B33"," ", " ", "B34"," ", " "],  # y=2

    [" ", "#", " ",  "#", "#", " ",  "#", " ", "#",  "#", " ", "#",  " ", "#", " ",  "#", " "],  # y=3
    [" ", "#", " ",  " ", " ", " ",  "#", " ", "#",  " ", " ", "#",  " ", "#", " ",  "#", " "],  # y=4

    [" ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " ", " ",  " ", " ", " ",  "#", " "],  # y=5

    [" ", "#", " ",  " ", "B35"," ",  "#", " ", "B36"," ", "#", " ",  "B37"," ", "#", " ", "B38"], # y=6

    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],  # y=7
    [" ", " ", " ",  " ", "#", " ",  " ", " ", "#",  " ", " ", " ",  "#", " ", " ",  " ", " "],  # y=8

    ["#", "#", " ",  "#", "#", " ",  "#", " ", " ",  " ", "#", " ",  "#", " ", "#",  "#", " "],  # y=9

    [" ", " ", " ",  " ", "#", " ",  "C", " ", "#",  " ", "#", " ",  " ", " ", "B",  " ", " "],  # y=10

    [" ", "#", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", " "],  # y=11
    [" ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " "],  # y=12

    [" ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " ", "#",  " ", "#", " ",  "#", " "],  # y=13
    [" ", "#", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  " ", " ", " ",  "#", " "],  # y=14

    [" ", " ", " ",  "#", "#", "#",  " ", "#", "#",  "#", " ", "#",  "#", "#", " ",  " ", " "],  # y=15
]


In [ ]:
import math
import heapq

# Estado = (x, y, piso)
# El edificio se puede inferir desde la grilla

def crear_estado(x, y, piso):
    return (x, y, piso)


#Identificar edificio automáticamente
def obtener_edificio(x, y, campus):
    valor = campus[y][x]

    if valor in ["B30","B31","B32","B33","B34",
                 "B35","B36","B37","B38"]:
        return valor
    elif valor == "C":
        return "Cafetería"
    elif valor == "B":
        return "Biblioteca"
    else:
        return "Pasillo"


#Heurística (distancia euclídea)
def heuristica(estado, goal):
    x, y, _ = estado
    xg, yg, _ = goal

    return math.sqrt((x - xg)**2 + (y - yg)**2)


#Acciones posibles
def es_transitable(x, y, campus):
    if x < 0 or y < 0:
        return False
    if y >= len(campus) or x >= len(campus[0]):
        return False
    if campus[y][x] == "#":
        return False
    return True


#sucesores
def sucesores(estado, campus):
    x, y, piso = estado
    edificio = obtener_edificio(x, y, campus)

    vecinos = []

    # Movimientos en el plano (costo 1)
    movimientos = [
        (0, -1, "arriba"),
        (0, 1, "abajo"),
        (-1, 0, "izquierda"),
        (1, 0, "derecha")
    ]

    for dx, dy, accion in movimientos:
        nx, ny = x + dx, y + dy
        if es_transitable(nx, ny, campus):
            vecinos.append(((nx, ny, piso), accion, 1))

    # Ascensor
    edificios_con_ascensor = ["B32","B33","B35","B37","B38"]

    if edificio in edificios_con_ascensor:
        if piso < 3:
            vecinos.append(((x, y, piso+1), "subir_ascensor", 3))
        if piso > 1:
            vecinos.append(((x, y, piso-1), "bajar_ascensor", 3))

    # Escaleras (en cualquier edificio que no sea pasillo)
    if edificio != "Pasillo":
        if piso < 8:
            vecinos.append(((x, y, piso+1), "subir_escaleras", 7 + 1))
        if piso > 1:
            vecinos.append(((x, y, piso-1), "bajar_escaleras", 5))

    return vecinos


# A*

def a_star(start, goal, campus):

    frontera = []
    heapq.heappush(frontera, (0, start))

    vino_de = {}
    costo_acumulado = {}

    vino_de[start] = None
    costo_acumulado[start] = 0

    while frontera:

        _, actual = heapq.heappop(frontera)

        if actual == goal:
            break

        for siguiente, accion, costo in sucesores(actual, campus):

            nuevo_costo = costo_acumulado[actual] + costo

            if siguiente not in costo_acumulado or nuevo_costo < costo_acumulado[siguiente]:

                costo_acumulado[siguiente] = nuevo_costo
                prioridad = nuevo_costo + heuristica(siguiente, goal)
                heapq.heappush(frontera, (prioridad, siguiente))

                vino_de[siguiente] = (actual, accion)

    # reconstruimos
    ruta, estados = reconstruir_camino(vino_de, start, goal)

    return ruta, estados, costo_acumulado.get(goal, None)


#reconstruir camino
def reconstruir_camino(vino_de, start, goal):

    actual = goal
    ruta = []
    estados = [goal]

    while actual != start:
        anterior, accion = vino_de[actual]
        ruta.append(accion)
        actual = anterior
        estados.append(actual)

    ruta.reverse()
    estados.reverse()

    return ruta, estados




# Prueba

In [10]:
start = (2, 2, 1)      # B30 piso 1
goal = (14, 10, 2)     # Biblioteca piso 2

ruta, estados, costo = a_star(start, goal, Campus)

print("Ruta:", ruta)
print("Secuencia de estados:")
for e in estados:
    print(e)

print("Costo total:", costo)


Ruta: ['derecha', 'derecha', 'derecha', 'derecha', 'derecha', 'derecha', 'derecha', 'derecha', 'abajo', 'abajo', 'abajo', 'derecha', 'abajo', 'derecha', 'subir_ascensor', 'derecha', 'abajo', 'abajo', 'abajo', 'abajo', 'derecha']
Secuencia de estados:
(2, 2, 1)
(3, 2, 1)
(4, 2, 1)
(5, 2, 1)
(6, 2, 1)
(7, 2, 1)
(8, 2, 1)
(9, 2, 1)
(10, 2, 1)
(10, 3, 1)
(10, 4, 1)
(10, 5, 1)
(11, 5, 1)
(11, 6, 1)
(12, 6, 1)
(12, 6, 2)
(13, 6, 2)
(13, 7, 2)
(13, 8, 2)
(13, 9, 2)
(13, 10, 2)
(14, 10, 2)
Costo total: 23
